In [3]:
import ecco_access as ea
import xarray as xr
import matplotlib.pyplot as plt
from os.path import join,expanduser

# import ecco_v4_py as ecco
import ecco_access as ea

In [4]:
#https://raw.githubusercontent.com/ECCO-GROUP/ECCO-ACCESS/refs/heads/main/varlist/v4r4/v4r4_nctiles_monthly_varlist.txt
#https://raw.githubusercontent.com/ECCO-GROUP/ECCO-ACCESS/refs/heads/main/varlist/v4r4/v4r4_nctiles_snapshots_varlist.txt

In [7]:
# download data and open xarray dataset
geometry_shortname = 'ECCO_L4_GEOMETRY_LLC0090GRID_V4R4'

ds_grid = xr.open_dataset("./data/ECCO_V4r4_PODAAC/GRID_GEOMETRY_ECCO_V4r4_native_llc0090.nc")
ds_grid = ds_grid.assign_coords(ds_grid.data_vars)

ssh_snapshot_shortname = 'ECCO_L4_SSH_LLC0090GRID_SNAPSHOT_V4R4'
ds_snapshot_dict = ea.ecco_podaac_to_xrdataset([ssh_snapshot_shortname,],\
                                        StartDate='1993-01-01',EndDate='1993-02',\
                                        mode='download',\
                                        download_root_dir=join('data','ECCO_V4r4_PODAAC'))

uvelmass_shortname= 'ECCO_L4_OCEAN_3D_VOLUME_FLUX_LLC0090GRID_MONTHLY_V4R4'
ds_monthly_dict = ea.ecco_podaac_to_xrdataset([uvelmass_shortname],\
                                        StartDate='1993-01-01',EndDate='1993-02',\
                                        mode='download',\
                                        download_root_dir=join('data','ECCO_V4r4_PODAAC'))
flux_shortname = "ECCO_L4_FRESH_FLUX_LLC0090GRID_MONTHLY_V4R4"
ds_flux_dict = ea.ecco_podaac_to_xrdataset([flux_shortname],\
                                        StartDate='1993-01-01',EndDate='1993-02',\
                                        mode='download',\
                                        download_root_dir=join('data','ECCO_V4r4_PODAAC'))

Download to directory data/ECCO_V4r4_PODAAC/ECCO_L4_SSH_LLC0090GRID_SNAPSHOT_V4R4

SEA_SURFACE_HEIGHT_snap_1993-02-01T000000_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading

SEA_SURFACE_HEIGHT_snap_1993-01-01T000000_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading

SEA_SURFACE_HEIGHT_snap_1993-03-01T000000_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading
DL Progress: 100%|########################| 3/3 [00:00<00:00, 83330.54it/s]

total downloaded: 0.0 Mb
avg download speed: 0.0 Mb/s
Time spent = 0.0016717910766601562 seconds


Download to directory data/ECCO_V4r4_PODAAC/ECCO_L4_OCEAN_3D_VOLUME_FLUX_LLC0090GRID_MONTHLY_V4R4

OCEAN_3D_VOLUME_FLUX_mon_mean_1993-01_ECCO_V4r4_native_llc0090.nc already exists, and force=False, not re-downloading
DL Progress: 100%|###########################| 2/2 [00:06<00:00,  3.32s/it]

total downloaded: 30.6 Mb
avg download speed: 4.61 Mb/s
Time spent = 6.64362716

In [8]:
snap_terms = ["ETAN"]
ds_budget_snap = xr.merge([v for v in ds_snapshot_dict.values()], compat='no_conflicts')[snap_terms]
ds_budget_snap = ds_budget_snap.rename({var:f"{var}_bounds" for var in ds_budget_snap.data_vars})
ds_budget_snap = ds_budget_snap.rename({"time":"time_bounds"})

monthly_terms = ["UVELMASS", "VVELMASS", "WVELMASS"]
ds_budget_monthly = xr.merge([v for v in ds_monthly_dict.values()], compat='no_conflicts')[monthly_terms]

flux_terms = ["oceFWflx"]
ds_flux_monthly = xr.merge([v for v in ds_flux_dict.values()], compat='no_conflicts')[flux_terms]


In [9]:
ds_budget = xr.merge([ds_grid, ds_budget_snap, ds_budget_monthly, ds_flux_monthly], compat='no_conflicts')
ds_budget.to_zarr("./data/ECCO_budget_terms.zarr", compute = True, mode = "w")

/Users/anthonymeza/miniforge3/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
